In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/aieducation_google/what-course-are-you-going-to-take/src/small_bench/checkpoints/pre_cell_10.pickle

In [ ]:
%%cudf.pandas.profile
### cell 10 ###

def extract_filter_features(df, col_name, threshold):
    """Extracts features for predicting execution time of filtering operation."""

    col = df[col_name]
    
    # In cudf, comparisons with nulls result in nulls, unlike pandas where they result in False.
    # We use fillna(False) to replicate the original pandas behavior in a vectorized way,
    # which avoids branching and an expensive dropna call.
    temp_df = pd.DataFrame({
        'nan_ratio': col.isna(),
        'target_selectivity': (col < threshold).fillna(False)
    })
    agg_results = temp_df.mean().to_dict()

    agg_results['num_rows'] = len(df)
    agg_results['dtype_str'] = str(col.dtype)

    return agg_results

extract_filter_features(course, "num_reviews", 10)